In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_access_management/ai_assistant_usage"
tgt_silver_table = "data_governance.silver_access_management.assistant_usage"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:

df = df.withColumn("account_id", trim(col("account_id"))) \
       .withColumn("workspace_id", trim(col("workspace_id"))) \
       .withColumn("event_id", trim(col("event_id"))) \
       .withColumn("initiated_by", lower(trim(col("initiated_by"))))

df = df.dropDuplicates(["event_id", "workspace_id"])

df = df.withColumn("event_year", year("event_time")) \
       .withColumn("event_month", month("event_time")) \
       .withColumn("event_day", dayofmonth("event_time")) \
       .withColumn("event_hour", hour("event_time"))

df = df.withColumn(
    "browser",
    regexp_extract("user_agent", "(Chrome|Firefox|Safari|Edge|Mozilla)", 1)
)

df = df.withColumn(
    "os",
    regexp_extract("user_agent", "(Windows|Mac OS X|Linux)", 1)
)


df = df.withColumn(
    "is_bot",
    when(col("user_agent").rlike("bot|crawler|spider"), True).otherwise(False)
)

df = df.filter(
    col("event_id").isNotNull() &
    col("workspace_id").isNotNull() &
    col("event_time").isNotNull()
)

window = Window.partitionBy("initiated_by").orderBy("event_time")

df = df.withColumn(
    "prev_event_time",
    lag("event_time").over(window)
)

In [0]:
df.display()

In [0]:

df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("event_year", "event_month", "event_day") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_access_management.assistant_usage;